In [1]:
from langchain_community.chat_message_histories import SQLChatMessageHistory
from dotenv import load_dotenv

load_dotenv()

chat_message_history = SQLChatMessageHistory(
    session_id="sql_history", connection="sqlite:///sqlite.db"
)

In [5]:
chat_message_history.add_user_message(
    "안녕? 만나서 반가워. 내 이름은 테디야. 나는 랭체인 개발자야. 앞으로 잘 부탁해!"
)

chat_message_history.add_ai_message("안녕 테디, 만나서 반가워. 나도 잘 부탁해!")


In [6]:
chat_message_history.messages

[HumanMessage(content='안녕? 만나서 반가워. 내 이름은 테디야. 나는 랭체인 개발자야. 아픙로 잘 부탁해!', additional_kwargs={}, response_metadata={}),
 AIMessage(content='안녕 테디, 만나서 반가워. 나도 잘 부탁해!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='안녕? 만나서 반가워. 내 이름은 테디야. 나는 랭체인 개발자야. 앞으로 잘 부탁해!', additional_kwargs={}, response_metadata={}),
 AIMessage(content='안녕 테디, 만나서 반가워. 나도 잘 부탁해!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [9]:
from langchain_core.prompts import(
    ChatPromptTemplate,
    MessagesPlaceholder,
)
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages(
    [
        ("system","You are a helpful assistant."),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human","{question}"),
    ]
)

chain = prompt | ChatOpenAI(model="gpt-4o-mini") | StrOutputParser()

In [10]:
def get_chat_history(user_id, conversation_id):
    return SQLChatMessageHistory(
        table_name=user_id,
        session_id=conversation_id,
        connection="sqlite:///sqlite.db",
    )

In [11]:
from langchain_core.runnables.utils import ConfigurableFieldSpec

config_fields = [
    ConfigurableFieldSpec(
        id="user_id",
        annotation=str,
        name="User ID",
        description="Unique identifier for a user.",
        default="",
        is_shared=True,
    ),
    ConfigurableFieldSpec(
        id="conversation_id",
        annotation=str,
        name="Conversation ID",
        description="Unique identifier for a conversation.",
        default="",
        is_shared=True,
    ),
]

chain_with_history = RunnableWithMessageHistory(
    chain,
    get_chat_history,
    input_messages_key="question",
    history_messages_key="chat_history",
    history_factory_config=config_fields,
)

d:\hanwha_0902\venv\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [12]:
config = {"configurable":{"user_id": "user1", "conversation_id":"conversation1"}}

In [13]:
chain_with_history.invoke({"question": "안녕 반가워 내 이름은 테디야"}, config)

'안녕하세요, 테디! 만나서 반가워요. 어떻게 도와드릴까요?'

In [14]:
chain_with_history.invoke({"question":"내 이름이 뭐라고?"},config)

'당신의 이름은 테디라고 하셨어요. 맞나요?'

In [16]:
config = {"configurable":{"user_id":"user1", "conversation_id": "conversation2"}}

chain_with_history.invoke({"question": "내 이름이 뭐라고 ?"},config)

'죄송하지만, 당신의 이름을 알 수 있는 정보가 없습니다. 어떤 이름을 원하시거나 다른 질문이 있으신가요?'

In [21]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.output_parsers import StrOutputParser
from langchain_teddynote import logging
from dotenv import load_dotenv

logging.langsmith("Ch05-Memory")
load_dotenv()

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "당신은 Question-Answering 챗봇입니다. 주어진 질문에 대한 답변을 제공해 주세요."
        ),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human","#Question:\n{question}"),
        
    ]
)

llm = ChatOpenAI(model="gpt-4o-mini")

chain = prompt | llm | StrOutputParser()

LangSmith 추적을 시작합니다.
[프로젝트명]
Ch05-Memory


In [22]:
store = {}

def get_session_history(session_ids):
    print(f"[대화 세션ID]: {session_ids}")
    if session_ids not in store:
        store[session_ids] = ChatMessageHistory()
    return store[session_ids]   

In [23]:
chain_with_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="question",
    history_messages_key="chat_history",
)

d:\hanwha_0902\venv\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [ ]:
chain_with_history.invoke(
    {"question":"나의 이름은 테디입니다."},
    config={"configurable":{"session_id":"abc123"}},
)